<a href="https://colab.research.google.com/github/sumitgithub24/Banking_Security/blob/main/Audio_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pydub noisereduce json-tricks

In [ ]:
import numpy as np
import os
from json_tricks import dump, load

from pydub import AudioSegment, effects
import librosa
import noisereduce as nr

import tensorflow as tf
import keras
import sklearn

In [ ]:
# 'emotions' list fix for classification purposes:
#     Classification values start from 0, Thus an 'n = n-1' operation has been executed for both RAVDESS and TESS databases:
def emotionfix(e_num):
    if e_num == "01":   return 0 # neutral
    elif e_num == "02": return 1 # calm
    elif e_num == "03": return 2 # happy
    elif e_num == "04": return 3 # sad
    elif e_num == "05": return 4 # angry
    elif e_num == "06": return 5 # fear
    elif e_num == "07": return 6 # disgust
    else:               return 7 # suprised

Maximum samples count for padding purposes.
sample_lengths = [] folder_path = 'Ravdess'

for subdir, dirs, files in os.walk(folder_path): for file in files: x, sr = librosa.load(path = os.path.join(subdir,file), sr = None) xt, index = librosa.effects.trim(x, top_db=30) sample_lengths.append(len(xt))

print('Maximum sample length:', np.max(sample_lengths))

In [5]:
from google.colab import files
uploaded = files.upload()

Saving Ravdess.zip to Ravdess.zip


In [10]:
import zipfile

zip_path = '/content/Ravdess.zip'  # Path to your ZIP file
extract_path = '/content/RAVDESS'  # Destination folder

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

    print("Unzipped successfully to:", extract_path)


Unzipped successfully to: /content/RAVDESS


In [11]:
# Maximum samples count for padding purposes.

sample_lengths = []
folder_path = 'RAVDESS/'

for subdir, dirs, files in os.walk(folder_path):
  for file in files:
      x, sr = librosa.load(path = os.path.join(subdir,file), sr = None)
      xt, index = librosa.effects.trim(x, top_db=30)
      sample_lengths.append(len(xt))

print('Maximum sample length:', np.max(sample_lengths))

Maximum sample length: 204288


In [12]:
import time
tic = time.perf_counter()

# Initialize data lists
rms = []
zcr = []
mfcc = []
emotions = []

# Initialize variables
total_length = 173056 # desired frame length for all of the audio samples.
frame_length = 2048
hop_length = 512

folder_path = 'RAVDESS/'

for subdir, dirs, files in os.walk(folder_path):
  for file in files:
    # Fetch the sample rate.
    _, sr = librosa.load(path = os.path.join(subdir,file), sr = None) # sr (the sample rate) is used for librosa's MFCCs. '_' is irrelevant.
    # Load the audio file.
    rawsound = AudioSegment.from_file(os.path.join(subdir,file))
    # Normalize the audio to +5.0 dBFS.
    normalizedsound = effects.normalize(rawsound, headroom = 0)
    # Transform the normalized audio to np.array of samples.
    normal_x = np.array(normalizedsound.get_array_of_samples(), dtype = 'float32')
    # Trim silence from the beginning and the end.
    xt, index = librosa.effects.trim(normal_x, top_db=30)
    #print(file,"\t", len(xt), "\t", rawsound.dBFS, "\t", normalizedsound.dBFS) #--QA purposes if needed--
    # Pad for duration equalization.
    padded_x = np.pad(xt, max(0, (total_length - len(xt))), 'constant')
    # Noise reduction.
    final_x = nr.reduce_noise(padded_x, sr=sr) #updated 03/03/22

    # Features extraction
    f1 = librosa.feature.rms(y=final_x, frame_length=frame_length, hop_length=hop_length)# Energy - Root Mean Square
    f2 = librosa.feature.zero_crossing_rate(y=final_x , frame_length=frame_length, hop_length=hop_length, center=True) # ZCR
    f3 = librosa.feature.mfcc(y=final_x, sr=sr, n_mfcc=13, hop_length = hop_length) # MFCC

    # Emotion extraction from the different databases
    name = file[6:8]

    # Filling the data lists
    rms.append(f1)
    zcr.append(f2)
    mfcc.append(f3)
    emotions.append(emotionfix(name))
toc = time.perf_counter()
print(f"Running time: {(toc - tic)/60:0.4f}")

Running time: 5.5331


In [13]:
# Determine the maximum length of sequences in `rms`
max_len = max(r.shape[1] for r in rms)
# Pad sequences in `rms` to have the same length
rms_padded = [np.pad(r, ((0, 0), (0, max_len - r.shape[1]))) for r in rms]

# Convert the list to a numpy array
rms = np.stack(rms_padded)
# Determine the maximum length of sequences in `zcr`
max_len = max(z.shape[1] for z in zcr)

# Pad sequences in `zcr` to have the same length
zcr_padded = [np.pad(z, ((0, 0), (0, max_len - z.shape[1]))) for z in zcr]

# Convert the list to a numpy array
zcr = np.stack(zcr_padded)
# Determine the maximum length of sequences in `mfcc`
max_len = max(m.shape[1] for m in mfcc)

# Pad sequences in `mfcc` to have the same length
mfcc_padded = [np.pad(m, ((0, 0), (0, max_len - m.shape[1]))) for m in mfcc]

# Convert the list to a numpy array
mfcc = np.stack(mfcc_padded)

In [14]:
# Adjusting features shape to the 3D format: (batch, timesteps, feature)

# making all in equal dimensions

f_rms = np.asarray(rms).astype('float32')
f_rms = np.swapaxes(f_rms,1,2)
f_zcr = np.asarray(zcr).astype('float32')
f_zcr = np.swapaxes(f_zcr,1,2)
f_mfccs = np.asarray(mfcc).astype('float32')
f_mfccs = np.swapaxes(f_mfccs,1,2)

print('ZCR shape:',f_zcr.shape)
print('RMS shape:',f_rms.shape)
print('MFCCs shape:',f_mfccs.shape)

ZCR shape: (1440, 577, 1)
RMS shape: (1440, 577, 1)
MFCCs shape: (1440, 577, 13)


In [15]:
# Concatenating all features to 'X' variable.
X = np.concatenate((f_zcr, f_rms, f_mfccs), axis=2)

# Preparing 'Y' as a 2D shaped variable.
Y = np.asarray(emotions).astype('int8')
Y = np.expand_dims(Y, axis=1)

In [16]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

Using RBF Kernel SVM to classify the data

In [17]:
from sklearn.svm import SVC
model = SVC(kernel='rbf', C=1, gamma='scale', random_state=42)

In [18]:
model.fit(X_train.reshape(X_train.shape[0], -1), y_train.ravel())

SVC(C=1, random_state=42)

In a Jupyter environment, please rerun this cell to show the HTML representation or trust the notebook.
On GitHub, the HTML representation is unable to render, please try loading this page with nbviewer.org.

In [19]:
# Make predictions
y_pred = model.predict(X_test.reshape(X_test.shape[0], -1))

# Evaluate performance
from sklearn.metrics import classification_report, accuracy_score
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred)*100, "%")

Classification Report:
              precision    recall  f1-score   support

           0       0.18      0.11      0.14        18
           1       0.40      0.55      0.46        42
           2       0.15      0.31      0.21        26
           3       0.16      0.09      0.12        44
           4       0.39      0.33      0.36        36
           5       0.25      0.14      0.18        44
           6       0.31      0.36      0.33        33
           7       0.48      0.51      0.49        45

    accuracy                           0.31       288
   macro avg       0.29      0.30      0.29       288
weighted avg       0.30      0.31      0.30       288

Accuracy: 31.25 %


Using LSTM to classify the data (RNN)

In [20]:
from keras.models import Sequential
from keras.layers import Dense, LSTM, Dropout, Flatten

# Define the model architecture
model = Sequential()
model.add(LSTM(128, input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Print the model summary
model.summary()

# Train the model
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 128)            │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 82,049 (320.50 KB)

 Trainable params: 82,049 (320.50 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 30s 760ms/step - accuracy: 0.1285 - loss: -8.7339 - val_accuracy: 0.1458 - val_loss: -81.6115
Epoch 2/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 37s 651ms/step - accuracy: 0.1122 - loss: -114.5306 - val_accuracy: 0.1458 - val_loss: -228.0912
Epoch 3/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 41s 660ms/step - accuracy: 0.1489 - loss: -253.7179 - val_accuracy: 0.1458 - val_loss: -430.0879
Epoch 4/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 24s 663ms/step - accuracy: 0.1390 - loss: -467.3275 - val_accuracy: 0.1458 - val_loss: -696.5955
Epoch 5/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 41s 668ms/step - accuracy: 0.1329 - loss: -743.1020 - val_accuracy: 0.1458 - val_loss: -1030.7606
Epoch 6/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 42s 686ms/step - accuracy: 0.1225 - loss: -1111.2876 - val_accuracy: 0.1458 - val_loss: -1439.5602
Epoch 7/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 39s 658ms/step - accuracy: 0.1292 - loss: -1487.9136 - val_accuracy: 0.1458 - val_loss: -1917.8550
Epoch 8/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 40s 643ms/step 

Using the RandomForestClassifier to classify the data

In [21]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.reshape(X_train.shape[0], -1))
X_test_scaled = scaler.transform(X_test.reshape(X_test.shape[0], -1))

# Initialize and train the Random Forest classifier
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(X_train_scaled, y_train)

# Make predictions
y_pred = rf_classifier.predict(X_test_scaled)

# Evaluate the model
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred)*100, "%")

/usr/local/lib/python3.11/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Classification Report:
              precision    recall  f1-score   support

           0       0.33      0.28      0.30        18
           1       0.50      0.76      0.60        42
           2       0.42      0.54      0.47        26
           3       0.40      0.23      0.29        44
           4       0.63      0.67      0.65        36
           5       0.47      0.36      0.41        44
           6       0.58      0.79      0.67        33
           7       0.56      0.42      0.48        45

    accuracy                           0.51       288
   macro avg       0.49      0.51      0.48       288
weighted avg       0.50      0.51      0.49       288

Accuracy: 50.69444444444444 %



Using the XGBoost Classifier to classify the data

In [22]:
from xgboost import XGBClassifier

# Standardize features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train.reshape(X_train.shape[0], -1))
X_test_scaled = scaler.transform(X_test.reshape(X_test.shape[0], -1))

# Initialize and train the XGBoost classifier
xgb_classifier = XGBClassifier(random_state=42)
xgb_classifier.fit(X_train_scaled, y_train)

# Make predictions
y_pred = xgb_classifier.predict(X_test_scaled)

In [23]:
# Evaluate the model
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred)*100, "%")

Classification Report:
              precision    recall  f1-score   support

           0       0.33      0.44      0.38        18
           1       0.56      0.64      0.60        42
           2       0.36      0.58      0.44        26
           3       0.52      0.39      0.44        44
           4       0.69      0.61      0.65        36
           5       0.50      0.43      0.46        44
           6       0.61      0.82      0.70        33
           7       0.81      0.49      0.61        45

    accuracy                           0.55       288
   macro avg       0.55      0.55      0.54       288
weighted avg       0.57      0.55      0.55       288

Accuracy: 54.513888888888886 %


Using CNN to classify the data

In [26]:
import tensorflow as tf
from tensorflow.keras import layers, models
# Define the CNN model
def create_cnn_model(input_shape):
    model = models.Sequential()
    model.add(layers.Conv1D(32, 3, activation='relu', input_shape=input_shape))
    model.add(layers.MaxPooling1D(2))
    model.add(layers.Conv1D(64, 3, activation='relu'))
    model.add(layers.MaxPooling1D(2))
    model.add(layers.Conv1D(128, 3, activation='relu'))
    model.add(layers.MaxPooling1D(2))
    model.add(layers.Conv1D(128, 3, activation='relu'))
    model.add(layers.MaxPooling1D(2))

    #flatten layer
    model.add(layers.Flatten())

    # Dense layers
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dense(8, activation='softmax'))  # Assuming 8 emotions
    return model
    # Input shape should match the shape of the concatenated features
input_shape = X.shape[1:]
# Create the CNN model
cnn_model = create_cnn_model(input_shape)
#compile model
cnn_model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
#train the model
history = cnn_model.fit(X, Y, epochs=30, batch_size=32, validation_split=0.2)




/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 7s 80ms/step - accuracy: 0.1646 - loss: 76.6857 - val_accuracy: 0.1424 - val_loss: 2.3769
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 5s 77ms/step - accuracy: 0.1963 - loss: 2.1393 - val_accuracy: 0.2222 - val_loss: 2.0280
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 6s 110ms/step - accuracy: 0.2431 - loss: 1.9362 - val_accuracy: 0.2326 - val_loss: 1.9559
Epoch 4/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 4s 118ms/step - accuracy: 0.3760 - loss: 1.7192 - val_accuracy: 0.2604 - val_loss: 1.9893
Epoch 5/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 86ms/step - accuracy: 0.4107 - loss: 1.5759 - val_accuracy: 0.2812 - val_loss: 1.9442
Epoch 6/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 4s 101ms/step - accuracy: 0.5100 - loss: 1.3346 - val_accuracy: 0.2396 - val_loss: 2.0062
Epoch 7/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 4s 79ms/step - accuracy: 0.5962 - loss: 1.1260 - val_accuracy: 0.2361 - val_loss: 2.0606
Epoch 8/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 5s 75ms/step - accuracy: 0.7085 - loss: 0.8809 - val_accuracy: 0.2569

In [27]:
# Evaluate the model
loss, accuracy = cnn_model.evaluate(X, Y)
print(f'Test Loss: {loss}, Test Accuracy: {accuracy*100}%')

45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9808 - loss: 0.1411
Test Loss: 1.013756275177002, Test Accuracy: 86.11111044883728%


In [28]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Define the CNN model
def create_cnn_model(input_shape):
    model = models.Sequential()

    # Convolutional layers
    model.add(layers.Conv1D(32, 3, activation='relu', input_shape=input_shape))
    model.add(layers.MaxPooling1D(2))
    model.add(layers.Conv1D(64, 3, activation='relu'))
    model.add(layers.MaxPooling1D(2))
    model.add(layers.Conv1D(128, 3, activation='relu'))
    model.add(layers.MaxPooling1D(2))
    model.add(layers.Conv1D(128, 3, activation='relu'))
    model.add(layers.MaxPooling1D(2))

    # Flatten layer
    model.add(layers.Flatten())

    # Dense layers
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dense(8, activation='softmax'))  # Assuming 8 emotions

    return model
 # Input shape should match the shape of the concatenated features
input_shape = X.shape[1:]
# Create the CNN model
cnn_model = create_cnn_model(input_shape)
# Compile the model
cnn_model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
# Train the model
history = cnn_model.fit(X, Y, epochs=30, batch_size=32, validation_split=0.2)

Epoch 1/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 10s 193ms/step - accuracy: 0.1428 - loss: 44.8563 - val_accuracy: 0.1528 - val_loss: 2.1967
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 6s 71ms/step - accuracy: 0.2315 - loss: 2.0202 - val_accuracy: 0.2604 - val_loss: 1.9285
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 6s 95ms/step - accuracy: 0.3144 - loss: 1.7975 - val_accuracy: 0.2778 - val_loss: 1.8696
Epoch 4/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 95ms/step - accuracy: 0.4173 - loss: 1.5561 - val_accuracy: 0.2812 - val_loss: 1.8826
Epoch 5/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 77ms/step - accuracy: 0.4773 - loss: 1.4432 - val_accuracy: 0.3021 - val_loss: 1.9494
Epoch 6/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 73ms/step - accuracy: 0.5588 - loss: 1.2280 - val_accuracy: 0.2778 - val_loss: 2.0317
Epoch 7/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 6s 105ms/step - accuracy: 0.5866 - loss: 1.0988 - val_accuracy: 0.3090 - val_loss: 2.1376
Epoch 8/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 86ms/step - accuracy: 0.6496 - loss: 0.9621 - val_accuracy: 0.2847

In [29]:
# Save the CNN model
cnn_model.save("emotion_cnn_model.h5")

In [30]:
# Load the CNN model
import tensorflow as tf
loaded_model = tf.keras.models.load_model("emotion_cnn_model.h5")
print(loaded_model.summary())

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_12 (Conv1D)              │ (None, 575, 32)        │         1,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_12 (MaxPooling1D) │ (None, 287, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_13 (Conv1D)              │ (None, 285, 64)        │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_13 (MaxPooling1D) │ (None, 142, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_14 (Conv1D)              │ (None, 140, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_14 (MaxPooling1D) │ (None, 70, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_15 (Conv1D)              │ (None, 68, 128)        │        49,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_15 (MaxPooling1D) │ (None, 34, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 4352)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 128)            │       557,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 647,626 (2.47 MB)

 Trainable params: 647,624 (2.47 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

None


In [33]:
import librosa
import numpy as np
import soundfile as sf
def preprocess_audio(audio, sr, target_length=575, n_mfcc=13):
  # Normalize audio
  audio = audio / np.max(np.abs(audio), axis=0)
  # Trim silence
  audio, _ = librosa.effects.trim(audio)
  # Pad or truncate to match the target length
  if len(audio) > target_length:
    audio = audio[:target_length]
  else:
    audio = np.pad(audio, (0, max(0, target_length - len(audio))), 'constant')

  # Extract MFCCs
  mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)

  # Calculate RMS
  rms = librosa.feature.rms(y=audio)

  # Calculate ZCR
  zcr = librosa.feature.zero_crossing_rate(y=audio)

  # Stack MFCC with RMS and ZCR along the feature axis
  features = np.vstack([mfcc, rms, zcr])

  # Pad or truncate the feature matrix to match the required time steps
  if features.shape[1] < target_length:
    features = np.pad(features, ((0, 0), (0, target_length - features.shape[1])), 'constant')
  elif features.shape[1] > target_length:
    features = features[:, :target_length]

  return features.T[np.newaxis, :, :]

# Example usage
audio, sr = sf.read('temp.wav')
features = preprocess_audio(audio, sr)
print("Shape of processed features:", features.shape)

Shape of processed features: (1, 575, 15)


/usr/local/lib/python3.11/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=575
  warnings.warn(


In [34]:
prediction = loaded_model.predict(features)
emotion_index = np.argmax(prediction)
emotions = ['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']
predicted_emotion = emotions[emotion_index]
print("Predicted emotion:", predicted_emotion)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step
Predicted emotion: happy
